# K-Fold Cross Validation Feature Extraction

Notebook ini mengotomatisasi feature extraction untuk semua 5 splits menggunakan K-Fold approach dengan rasio **60:40** (training:testing).

## Output yang Dihasilkan:
- **20 file CSV**: X0 dan y0 untuk train dan test di setiap split (5 splits × 4 files)
- **Features**: gazeX, gazeY, kecepatan, direction, acceleration, cumulative-distance, displacement, stddev_2pop, linearity_index
- **Split Ratio**: 60% training data, 40% testing data untuk setiap split
- **Automated processing**: Semua preprocessing dan feature engineering dalam satu run

## 1. Import Libraries dan Setup

In [15]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import warnings
from sklearn.model_selection import KFold
warnings.filterwarnings('ignore')

print("=== K-Fold Feature Extraction Setup ===")
print("Libraries imported successfully!")

# Buat folder output jika belum ada
output_dir_splits = 'Dataset/Split'
output_dir_features = 'feature-engineering-kfold'
os.makedirs(output_dir_splits, exist_ok=True)
os.makedirs(output_dir_features, exist_ok=True)

print(f"Split data output directory: {output_dir_splits}")
print(f"Feature engineering output directory: {output_dir_features}")

# Load data asli
print("\n📊 Loading original dataset...")
original_file = 'truncated_dataset-seqglo.csv'
if os.path.exists(original_file):
    df_original = pd.read_csv(original_file)
    print(f"✅ Data loaded: {len(df_original):,} rows, {df_original.shape[1]} columns")
    print(f"   Unique participants: {df_original['nama'].nunique()}")
    print(f"   Label distribution: {df_original['label'].value_counts().to_dict()}")
    print(f"   Columns: {list(df_original.columns)}")
    
    # Quick data quality check
    print(f"\n🔍 Data Quality Check:")
    print(f"   Missing values: {df_original.isnull().sum().sum()}")
    print(f"   Duplicate rows: {df_original.duplicated().sum()}")
    
else:
    print(f"❌ File {original_file} not found!")
    print("   Please ensure the file is in the current directory")

=== K-Fold Feature Extraction Setup ===
Libraries imported successfully!
Split data output directory: Dataset/Split
Feature engineering output directory: feature-engineering-kfold

📊 Loading original dataset...
✅ Data loaded: 250,320 rows, 5 columns
   Unique participants: 70
   Label distribution: {2: 125160, 1: 125160}
   Columns: ['nama', 'time', 'gazeX', 'gazeY', 'label']

🔍 Data Quality Check:
   Missing values: 0
   Duplicate rows: 0


## 2. Data Splitting Functions

In [16]:
def create_balanced_participant_splits(df, n_splits=5, train_ratio=0.6, random_state=42):
    """
    Membuat 5-fold cross validation splits berdasarkan participant
    dengan menggunakan rasio train:test = 60:40
    """
    print("=== Creating Balanced Participant Splits (60:40 Train:Test) ===")
    
    # Hitung distribusi label per participant
    participant_labels = df.groupby('nama')['label'].agg(['mean', 'count']).reset_index()
    participant_labels['dominant_label'] = (participant_labels['mean'] > 1.5).astype(int) + 1
    
    print(f"Participant analysis:")
    print(f"  Total participants: {len(participant_labels)}")
    print(f"  Label 1 dominant: {sum(participant_labels['dominant_label'] == 1)}")
    print(f"  Label 2 dominant: {sum(participant_labels['dominant_label'] == 2)}")
    
    # Pisahkan participants berdasarkan dominant label
    label1_participants = participant_labels[participant_labels['dominant_label'] == 1]['nama'].tolist()
    label2_participants = participant_labels[participant_labels['dominant_label'] == 2]['nama'].tolist()
    
    # Custom split untuk rasio 60:40
    from sklearn.model_selection import train_test_split
    import numpy as np
    np.random.seed(random_state)
    
    splits_info = []
    
    # Untuk setiap fold, buat split dengan rasio 60:40
    for fold_idx in range(n_splits):
        # Split label 1 participants dengan rasio 60:40
        train_participants_1, test_participants_1 = train_test_split(
            label1_participants, 
            train_size=train_ratio, 
            random_state=random_state + fold_idx,
            shuffle=True
        )
        
        # Split label 2 participants dengan rasio 60:40
        train_participants_2, test_participants_2 = train_test_split(
            label2_participants, 
            train_size=train_ratio, 
            random_state=random_state + fold_idx,
            shuffle=True
        )
        
        # Gabungkan participants
        train_participants = train_participants_1 + train_participants_2
        test_participants = test_participants_1 + test_participants_2
        
        # Buat dataframe untuk split ini
        train_df = df[df['nama'].isin(train_participants)].copy()
        test_df = df[df['nama'].isin(test_participants)].copy()
        
        # Simpan split data
        split_num = fold_idx + 1
        train_file = f'{output_dir_splits}/train_split_{split_num}.csv'
        test_file = f'{output_dir_splits}/test_split_{split_num}.csv'
        
        train_df.to_csv(train_file, index=False)
        test_df.to_csv(test_file, index=False)
        
        # Hitung statistik
        train_label_dist = train_df['label'].value_counts().to_dict()
        test_label_dist = test_df['label'].value_counts().to_dict()
        
        # Hitung rasio aktual
        total_samples = len(train_df) + len(test_df)
        actual_train_ratio = len(train_df) / total_samples
        actual_test_ratio = len(test_df) / total_samples
        
        split_info = {
            'split': split_num,
            'train_participants': len(train_participants),
            'test_participants': len(test_participants),
            'train_samples': len(train_df),
            'test_samples': len(test_df),
            'train_labels': train_label_dist,
            'test_labels': test_label_dist,
            'train_file': train_file,
            'test_file': test_file,
            'actual_train_ratio': actual_train_ratio,
            'actual_test_ratio': actual_test_ratio
        }
        splits_info.append(split_info)
        
        print(f"\nSplit {split_num} created:")
        print(f"   Train: {len(train_participants)} participants, {len(train_df):,} samples ({actual_train_ratio:.1%})")
        print(f"   Test: {len(test_participants)} participants, {len(test_df):,} samples ({actual_test_ratio:.1%})")
        print(f"   Train labels: {train_label_dist}")
        print(f"   Test labels: {test_label_dist}")
    
    return splits_info

# Create splits dari data asli dengan rasio 60:40
print("\nCreating K-Fold splits from original data (60% train, 40% test)...")
if 'df_original' in locals():
    splits_data = create_balanced_participant_splits(df_original, n_splits=5, train_ratio=0.6)
    print(f"\nAll 5 splits created successfully with 60:40 ratio!")
    print(f"   Output folder: {output_dir_splits}")
else:
    print("Original data not loaded, cannot create splits")


Creating K-Fold splits from original data (60% train, 40% test)...
=== Creating Balanced Participant Splits (60:40 Train:Test) ===
Participant analysis:
  Total participants: 70
  Label 1 dominant: 35
  Label 2 dominant: 35

Split 1 created:
   Train: 42 participants, 150,192 samples (60.0%)
   Test: 28 participants, 100,128 samples (40.0%)
   Train labels: {2: 75096, 1: 75096}
   Test labels: {2: 50064, 1: 50064}

Split 1 created:
   Train: 42 participants, 150,192 samples (60.0%)
   Test: 28 participants, 100,128 samples (40.0%)
   Train labels: {2: 75096, 1: 75096}
   Test labels: {2: 50064, 1: 50064}

Split 2 created:
   Train: 42 participants, 150,192 samples (60.0%)
   Test: 28 participants, 100,128 samples (40.0%)
   Train labels: {2: 75096, 1: 75096}
   Test labels: {2: 50064, 1: 50064}

Split 2 created:
   Train: 42 participants, 150,192 samples (60.0%)
   Test: 28 participants, 100,128 samples (40.0%)
   Train labels: {2: 75096, 1: 75096}
   Test labels: {2: 50064, 1: 50064}

## 3. Feature Engineering Functions

In [17]:
def extract_features(df):    
    # Buat copy untuk menghindari warning
    df_features = df.copy()
    
    # Konversi ke numerik
    df_features['gazeX'] = pd.to_numeric(df_features['gazeX'], errors='coerce')
    df_features['gazeY'] = pd.to_numeric(df_features['gazeY'], errors='coerce')
    
    # 1. VELOCITY & DIRECTION
    df_features['delta_gazeX'] = df_features['gazeX'].diff()
    df_features['delta_gazeY'] = df_features['gazeY'].diff()
    df_features['squared_sum'] = df_features['delta_gazeX']**2 + df_features['delta_gazeY']**2
    df_features['kecepatan'] = (df_features['squared_sum'] / 0.01667)**0.5
    df_features['direction'] = np.arctan2(df_features['delta_gazeY'], df_features['delta_gazeX'])
    
    # 2. ACCELERATION
    df_features['kecepatanX'] = df_features['delta_gazeX'] / 0.01667
    df_features['kecepatanY'] = df_features['delta_gazeY'] / 0.01667
    df_features['delta_vX'] = df_features['kecepatanX'].diff()
    df_features['delta_vY'] = df_features['kecepatanY'].diff()
    dt = 0.01667
    df_features['acceleration'] = np.sqrt((df_features['delta_vX'] / dt) ** 2 + (df_features['delta_vY'] / dt) ** 2)
    
    # 3. CUMULATIVE DISTANCE
    df_features['squared_diff'] = df_features['delta_gazeX']**2 + df_features['delta_gazeY']**2
    df_features['cumulative-distance'] = np.sqrt(df_features['squared_diff'].cumsum())
    
    # 4. DISPLACEMENT
    df_features['deltaX'] = df_features['gazeX'].diff().shift(-1)
    df_features['deltaY'] = df_features['gazeY'].diff().shift(-1)
    df_features['displacement'] = np.sqrt(df_features['deltaX']**2 + df_features['deltaY']**2)
    
    # 5. STANDARD DEVIATION (sliding window)
    window = 2
    stddev_values = np.zeros(len(df_features))
    
    for i in range(0, len(df_features) - window + 1, window):
        window_x = df_features['gazeX'].iloc[i:i+window]
        window_y = df_features['gazeY'].iloc[i:i+window]
        
        std_x = np.std(window_x, ddof=0)
        std_y = np.std(window_y, ddof=0)
        std_combined = np.sqrt(std_x**2 + std_y**2)
        
        stddev_values[i:i+window] = std_combined
    
    df_features['stddev_2pop'] = stddev_values
    
    # 6. LINEARITY INDEX
    df_features['linearity_index'] = df_features.apply(
        lambda row: row['displacement'] / row['cumulative-distance'] if row['cumulative-distance'] != 0 else 0,
        axis=1
    )
    
    # Fill NaN values dengan 0
    feature_columns = ['kecepatan', 'direction', 'acceleration', 'cumulative-distance', 
                      'displacement', 'stddev_2pop', 'linearity_index']
    for col in feature_columns:
        df_features[col] = df_features[col].fillna(0)
    
    # Drop kolom temporary
    df_features = df_features.drop(columns=[
        'delta_gazeX', 'delta_gazeY', 'squared_diff', 'squared_sum', 
        'kecepatanX', 'kecepatanY', 'delta_vX', 'delta_vY', 'deltaX', 'deltaY'
    ])
    
    # Pisahkan features dan labels
    X = df_features.drop(['nama', 'time', 'label'], axis=1)
    y = df_features['label']
    
    print(f"  Features extracted: {X.shape[1]} features, {len(X)} samples")
    print(f"  Feature names: {list(X.columns)}")
    
    return X, y

## 4. Complete K-Fold Pipeline

In [18]:
def process_all_splits():
    """
    Proses feature extraction untuk semua 5 splits yang sudah dibuat
    """
    print("=== Starting K-Fold Feature Extraction ===\n")
    
    # Summary statistics
    total_files_created = 0
    processing_summary = []
    
    # Process setiap split
    for split_num in tqdm(range(1, 6), desc="Processing splits"):        
        try:
            # Load training data
            train_file = f'{output_dir_splits}/train_split_{split_num}.csv'
            test_file = f'{output_dir_splits}/test_split_{split_num}.csv'
            
            if not os.path.exists(train_file) or not os.path.exists(test_file):
                print(f"Split {split_num} files not found!")
                print(f"   Expected: {train_file}")
                print(f"   Expected: {test_file}")
                continue
            
            # Load data
            train_df = pd.read_csv(train_file)
            test_df = pd.read_csv(test_file)
            
            print(f"\nProcessing Split {split_num}:")
            print(f" Train: {len(train_df):,} rows, Test: {len(test_df):,} rows")
            
            # Extract features untuk training data
            print(f"   Extracting training features...")
            X_train, y_train = extract_features(train_df)
            
            # Extract features untuk testing data  
            print(f"   Extracting testing features...")
            X_test, y_test = extract_features(test_df)
            
            # Save files ke folder feature engineering
            train_X_file = f'{output_dir_features}/X0-train-split{split_num}-seqglo-truncate.csv'
            train_y_file = f'{output_dir_features}/y0-train-split{split_num}-seqglo-truncate.csv'
            test_X_file = f'{output_dir_features}/X0-test-split{split_num}-seqglo-truncate.csv'
            test_y_file = f'{output_dir_features}/y0-test-split{split_num}-seqglo-truncate.csv'
            
            X_train.to_csv(train_X_file, index=False)
            y_train.to_csv(train_y_file, index=False)
            X_test.to_csv(test_X_file, index=False)
            y_test.to_csv(test_y_file, index=False)
            
            # Update statistics
            total_files_created += 4
            
            split_summary = {
                'split': split_num,
                'train_samples': len(X_train),
                'test_samples': len(X_test),
                'features': X_train.shape[1],
                'train_labels': y_train.value_counts().to_dict(),
                'test_labels': y_test.value_counts().to_dict()
            }
            processing_summary.append(split_summary)
            
            print(f"   Split {split_num} completed successfully!")
            print(f"      Files saved to: {output_dir_features}/")
            
        except Exception as e:
            print(f"   Error processing Split {split_num}: {str(e)}")
            continue
    
    return total_files_created, processing_summary

# Run the processing
print("Starting feature extraction pipeline...")
files_created, summary = process_all_splits()

print(f"\n" + "=" * 60)
print("K-FOLD FEATURE EXTRACTION COMPLETED!")
print("=" * 60)
print(f"Total files created: {files_created}")
print(f"Expected files: 20 (5 splits × 4 files per split)")
if files_created > 0:
    print(f"Success rate: {(files_created/20)*100:.1f}%")
else:
    print("No files created - check if data splits exist")

Starting feature extraction pipeline...
=== Starting K-Fold Feature Extraction ===



Processing splits:   0%|          | 0/5 [00:00<?, ?it/s]


Processing Split 1:
 Train: 150,192 rows, Test: 100,128 rows
   Extracting training features...
  Features extracted: 9 features, 150192 samples
  Feature names: ['gazeX', 'gazeY', 'kecepatan', 'direction', 'acceleration', 'cumulative-distance', 'displacement', 'stddev_2pop', 'linearity_index']
   Extracting testing features...
  Features extracted: 9 features, 150192 samples
  Feature names: ['gazeX', 'gazeY', 'kecepatan', 'direction', 'acceleration', 'cumulative-distance', 'displacement', 'stddev_2pop', 'linearity_index']
   Extracting testing features...
  Features extracted: 9 features, 100128 samples
  Feature names: ['gazeX', 'gazeY', 'kecepatan', 'direction', 'acceleration', 'cumulative-distance', 'displacement', 'stddev_2pop', 'linearity_index']
  Features extracted: 9 features, 100128 samples
  Feature names: ['gazeX', 'gazeY', 'kecepatan', 'direction', 'acceleration', 'cumulative-distance', 'displacement', 'stddev_2pop', 'linearity_index']


Processing splits:  20%|██        | 1/5 [00:20<01:22, 20.54s/it]

   Split 1 completed successfully!
      Files saved to: feature-engineering-kfold/

Processing Split 2:
 Train: 150,192 rows, Test: 100,128 rows
   Extracting training features...
  Features extracted: 9 features, 150192 samples
  Feature names: ['gazeX', 'gazeY', 'kecepatan', 'direction', 'acceleration', 'cumulative-distance', 'displacement', 'stddev_2pop', 'linearity_index']
   Extracting testing features...
  Features extracted: 9 features, 150192 samples
  Feature names: ['gazeX', 'gazeY', 'kecepatan', 'direction', 'acceleration', 'cumulative-distance', 'displacement', 'stddev_2pop', 'linearity_index']
   Extracting testing features...
  Features extracted: 9 features, 100128 samples
  Feature names: ['gazeX', 'gazeY', 'kecepatan', 'direction', 'acceleration', 'cumulative-distance', 'displacement', 'stddev_2pop', 'linearity_index']
  Features extracted: 9 features, 100128 samples
  Feature names: ['gazeX', 'gazeY', 'kecepatan', 'direction', 'acceleration', 'cumulative-distance', '

...

   Split 3 completed successfully!
      Files saved to: feature-engineering-kfold/

Processing Split 4:
 Train: 150,192 rows, Test: 100,128 rows
   Extracting training features...
  Features extracted: 9 features, 150192 samples
  Feature names: ['gazeX', 'gazeY', 'kecepatan', 'direction', 'acceleration', 'cumulative-distance', 'displacement', 'stddev_2pop', 'linearity_index']
   Extracting testing features...
  Features extracted: 9 features, 150192 samples
  Feature names: ['gazeX', 'gazeY', 'kecepatan', 'direction', 'acceleration', 'cumulative-distance', 'displacement', 'stddev_2pop', 'linearity_index']
   Extracting testing features...
  Features extracted: 9 features, 100128 samples
  Feature names: ['gazeX', 'gazeY', 'kecepatan', 'direction', 'acceleration', 'cumulative-distance', 'displacement', 'stddev_2pop', 'linearity_index']
  Features extracted: 9 features, 100128 samples
  Feature names: ['gazeX', 'gazeY', 'kecepatan', 'direction', 'acceleration', 'cumulative-distance', '

Processing splits:  80%|████████  | 4/5 [01:28<00:22, 22.40s/it]

   Split 4 completed successfully!
      Files saved to: feature-engineering-kfold/

Processing Split 5:
 Train: 150,192 rows, Test: 100,128 rows
   Extracting training features...
  Features extracted: 9 features, 150192 samples
  Feature names: ['gazeX', 'gazeY', 'kecepatan', 'direction', 'acceleration', 'cumulative-distance', 'displacement', 'stddev_2pop', 'linearity_index']
   Extracting testing features...
  Features extracted: 9 features, 150192 samples
  Feature names: ['gazeX', 'gazeY', 'kecepatan', 'direction', 'acceleration', 'cumulative-distance', 'displacement', 'stddev_2pop', 'linearity_index']
   Extracting testing features...
  Features extracted: 9 features, 100128 samples
  Feature names: ['gazeX', 'gazeY', 'kecepatan', 'direction', 'acceleration', 'cumulative-distance', 'displacement', 'stddev_2pop', 'linearity_index']
  Features extracted: 9 features, 100128 samples
  Feature names: ['gazeX', 'gazeY', 'kecepatan', 'direction', 'acceleration', 'cumulative-distance', '

Processing splits: 100%|██████████| 5/5 [01:52<00:00, 22.43s/it]

   Split 5 completed successfully!
      Files saved to: feature-engineering-kfold/

K-FOLD FEATURE EXTRACTION COMPLETED!
Total files created: 20
Expected files: 20 (5 splits × 4 files per split)
Success rate: 100.0%


## 4. Verification dan Summary

In [19]:
# Tampilkan summary detail
print("\n📊 DETAILED PROCESSING SUMMARY:")
print("=" * 80)

for split_info in summary:
    split_num = split_info['split']
    print(f"\n🔸 Split {split_num}:")
    print(f"   Training samples: {split_info['train_samples']:,}")
    print(f"   Testing samples: {split_info['test_samples']:,}")
    
    # Hitung dan tampilkan rasio aktual
    total_samples = split_info['train_samples'] + split_info['test_samples']
    train_pct = (split_info['train_samples'] / total_samples) * 100
    test_pct = (split_info['test_samples'] / total_samples) * 100
    print(f"   Actual ratio: {train_pct:.1f}% train, {test_pct:.1f}% test")
    
    print(f"   Features count: {split_info['features']}")
    print(f"   Train labels: {split_info['train_labels']}")
    print(f"   Test labels: {split_info['test_labels']}")

# Hitung total statistics
total_train_samples = sum([s['train_samples'] for s in summary])
total_test_samples = sum([s['test_samples'] for s in summary])
avg_train_per_split = total_train_samples / len(summary) if summary else 0
avg_test_per_split = total_test_samples / len(summary) if summary else 0

print(f"\n📈 OVERALL STATISTICS:")
print(f"   Total training samples across all splits: {total_train_samples:,}")
print(f"   Total testing samples across all splits: {total_test_samples:,}")
print(f"   Average training samples per split: {avg_train_per_split:,.0f}")
print(f"   Average testing samples per split: {avg_test_per_split:,.0f}")

if summary:
    print(f"   Features per dataset: {summary[0]['features']}")
    print(f"   Feature names: gazeX, gazeY, kecepatan, direction, acceleration, cumulative-distance, displacement, stddev_2pop, linearity_index")


📊 DETAILED PROCESSING SUMMARY:

🔸 Split 1:
   Training samples: 150,192
   Testing samples: 100,128
   Actual ratio: 60.0% train, 40.0% test
   Features count: 9
   Train labels: {2: 75096, 1: 75096}
   Test labels: {2: 50064, 1: 50064}

🔸 Split 2:
   Training samples: 150,192
   Testing samples: 100,128
   Actual ratio: 60.0% train, 40.0% test
   Features count: 9
   Train labels: {2: 75096, 1: 75096}
   Test labels: {2: 50064, 1: 50064}

🔸 Split 3:
   Training samples: 150,192
   Testing samples: 100,128
   Actual ratio: 60.0% train, 40.0% test
   Features count: 9
   Train labels: {2: 75096, 1: 75096}
   Test labels: {2: 50064, 1: 50064}

🔸 Split 4:
   Training samples: 150,192
   Testing samples: 100,128
   Actual ratio: 60.0% train, 40.0% test
   Features count: 9
   Train labels: {2: 75096, 1: 75096}
   Test labels: {2: 50064, 1: 50064}

🔸 Split 5:
   Training samples: 150,192
   Testing samples: 100,128
   Actual ratio: 60.0% train, 40.0% test
   Features count: 9
   Train labe

## 5. File Verification

In [20]:
# Verifikasi file yang telah dibuat
print("\n🔍 FILE VERIFICATION:")
print("=" * 50)

expected_files = []
for split_num in range(1, 6):
    expected_files.extend([
        f'{output_dir_features}/X0-train-split{split_num}-seqglo-truncate.csv',
        f'{output_dir_features}/y0-train-split{split_num}-seqglo-truncate.csv',
        f'{output_dir_features}/X0-test-split{split_num}-seqglo-truncate.csv',
        f'{output_dir_features}/y0-test-split{split_num}-seqglo-truncate.csv'
    ])

existing_files = 0
missing_files = []

for file_path in expected_files:
    if os.path.exists(file_path):
        file_size = os.path.getsize(file_path) / (1024 * 1024)  # MB
        print(f"✅ {file_path} ({file_size:.2f} MB)")
        existing_files += 1
    else:
        print(f"❌ {file_path} - MISSING")
        missing_files.append(file_path)

print(f"\n📋 VERIFICATION SUMMARY:")
print(f"   Files found: {existing_files}/{len(expected_files)}")
if len(expected_files) > 0:
    print(f"   Success rate: {(existing_files/len(expected_files))*100:.1f}%")

if missing_files:
    print(f"   Missing files: {len(missing_files)}")
    for missing in missing_files:
        print(f"     - {missing}")
else:
    print("   🎉 All files created successfully!")

# Additional verification untuk split files
print(f"\n🔍 SPLIT FILES VERIFICATION:")
print("=" * 40)
split_files_exist = 0
for split_num in range(1, 6):
    train_file = f'{output_dir_splits}/train_split_{split_num}.csv'
    test_file = f'{output_dir_splits}/test_split_{split_num}.csv'
    
    if os.path.exists(train_file) and os.path.exists(test_file):
        print(f"✅ Split {split_num}: train & test files exist")
        split_files_exist += 1
    else:
        print(f"❌ Split {split_num}: missing files")

print(f"\nSplit files status: {split_files_exist}/5 complete")


🔍 FILE VERIFICATION:
✅ feature-engineering-kfold/X0-train-split1-seqglo-truncate.csv (20.04 MB)
✅ feature-engineering-kfold/y0-train-split1-seqglo-truncate.csv (0.43 MB)
✅ feature-engineering-kfold/X0-test-split1-seqglo-truncate.csv (13.25 MB)
✅ feature-engineering-kfold/y0-test-split1-seqglo-truncate.csv (0.29 MB)
✅ feature-engineering-kfold/X0-train-split2-seqglo-truncate.csv (19.95 MB)
✅ feature-engineering-kfold/y0-train-split2-seqglo-truncate.csv (0.43 MB)
✅ feature-engineering-kfold/X0-test-split2-seqglo-truncate.csv (13.35 MB)
✅ feature-engineering-kfold/y0-test-split2-seqglo-truncate.csv (0.29 MB)
✅ feature-engineering-kfold/X0-train-split3-seqglo-truncate.csv (20.02 MB)
✅ feature-engineering-kfold/y0-train-split3-seqglo-truncate.csv (0.43 MB)
✅ feature-engineering-kfold/X0-test-split3-seqglo-truncate.csv (13.27 MB)
✅ feature-engineering-kfold/y0-test-split3-seqglo-truncate.csv (0.29 MB)
✅ feature-engineering-kfold/X0-train-split4-seqglo-truncate.csv (19.96 MB)
✅ feature-engin

In [21]:
# FINAL CONFIRMATION: Files Location
print("🎯 FILES LOCATION SUMMARY:")
print("=" * 60)

print(f"\n📁 Original Split Files (Dataset/Split): {len(os.listdir('Dataset/Split'))} files")
for file in sorted(os.listdir('Dataset/Split')):
    print(f"   {file}")

print(f"\n📁 Feature Engineering Files (feature-engineering): {len(os.listdir('feature-engineering'))} files")
for file in sorted(os.listdir('feature-engineering')):
    print(f"   {file}")

print(f"\n✅ TOTAL FILES CREATED: {len(os.listdir('feature-engineering'))} files")
print("✅ All 20 files (X0 and y0 for train/test across 5 splits) successfully created!")
print("\n🔧 If you can't see them in VS Code:")
print("   1. Refresh VS Code Explorer (Ctrl+Shift+E)")
print("   2. Navigate to 'feature-engineering' folder")
print("   3. Look for files ending with '-seqglo-truncate.csv'")

🎯 FILES LOCATION SUMMARY:

📁 Original Split Files (Dataset/Split): 10 files
   test_split_1.csv
   test_split_2.csv
   test_split_3.csv
   test_split_4.csv
   test_split_5.csv
   train_split_1.csv
   train_split_2.csv
   train_split_3.csv
   train_split_4.csv
   train_split_5.csv

📁 Feature Engineering Files (feature-engineering): 20 files
   X0-test-split1-seqglo-truncate.csv
   X0-test-split2-seqglo-truncate.csv
   X0-test-split3-seqglo-truncate.csv
   X0-test-split4-seqglo-truncate.csv
   X0-test-split5-seqglo-truncate.csv
   X0-train-split1-seqglo-truncate.csv
   X0-train-split2-seqglo-truncate.csv
   X0-train-split3-seqglo-truncate.csv
   X0-train-split4-seqglo-truncate.csv
   X0-train-split5-seqglo-truncate.csv
   y0-test-split1-seqglo-truncate.csv
   y0-test-split2-seqglo-truncate.csv
   y0-test-split3-seqglo-truncate.csv
   y0-test-split4-seqglo-truncate.csv
   y0-test-split5-seqglo-truncate.csv
   y0-train-split1-seqglo-truncate.csv
   y0-train-split2-seqglo-truncate.csv
   y0

## 6. Sample Data Preview

In [22]:
# Preview sample data dari split 1
if os.path.exists(f'{output_dir_features}/X0-train-split1-seqglo-truncate.csv'):
    print("\n👀 SAMPLE DATA PREVIEW (Split 1 - Training):")
    print("=" * 60)
    
    sample_X = pd.read_csv(f'{output_dir_features}/X0-train-split1-seqglo-truncate.csv')
    sample_y = pd.read_csv(f'{output_dir_features}/y0-train-split1-seqglo-truncate.csv')
    
    print(f"Features shape: {sample_X.shape}")
    print(f"Labels shape: {sample_y.shape}")
    print(f"\nFeature columns: {list(sample_X.columns)}")
    print(f"\nFirst 5 rows of features:")
    print(sample_X.head())
    print(f"\nLabel distribution:")
    print(sample_y['label'].value_counts())
    
    # Basic statistics
    print(f"\n📊 FEATURE STATISTICS:")
    print(sample_X.describe())
    
    # Original data comparison
    if 'df_original' in locals():
        print(f"\n📈 DATA COMPARISON:")
        print(f"   Original dataset: {len(df_original):,} rows")
        print(f"   Split 1 train: {len(sample_X):,} rows")
        print(f"   Reduction ratio: {len(sample_X)/len(df_original)*100:.1f}%")
else:
    print("❌ Sample files not found for preview")
    print(f"   Looking for: {output_dir_features}/X0-train-split1-seqglo-truncate.csv")


👀 SAMPLE DATA PREVIEW (Split 1 - Training):


Features shape: (150192, 9)
Labels shape: (150192, 1)

Feature columns: ['gazeX', 'gazeY', 'kecepatan', 'direction', 'acceleration', 'cumulative-distance', 'displacement', 'stddev_2pop', 'linearity_index']

First 5 rows of features:
      gazeX     gazeY  kecepatan  direction  acceleration  \
0  902.3616  863.3844        0.0        0.0           0.0   
1  902.3616  863.3844        0.0        0.0           0.0   
2  902.3616  863.3844        0.0        0.0           0.0   
3  902.3616  863.3844        0.0        0.0           0.0   
4  902.3616  863.3844        0.0        0.0           0.0   

   cumulative-distance  displacement  stddev_2pop  linearity_index  
0                  0.0           0.0          0.0              0.0  
1                  0.0           0.0          0.0              0.0  
2                  0.0           0.0          0.0              0.0  
3                  0.0           0.0          0.0              0.0  
4                  0.0           0.0          0.0      

## 🎯 Next Steps

Setelah K-Fold feature extraction selesai, Anda dapat:

1. **Load data untuk training LSTM:**
   ```python
   # Contoh untuk split 1
   X_train = pd.read_csv('feature-engineering/X0-train-split1-seqglo-truncate.csv')
   X_test = pd.read_csv('feature-engineering/X0-test-split1-seqglo-truncate.csv')
   y_train = pd.read_csv('feature-engineering/y0-train-split1-seqglo-truncate.csv')
   y_test = pd.read_csv('feature-engineering/y0-test-split1-seqglo-truncate.csv')
   ```

2. **Implementasi cross-validation training:**
   - Train model untuk setiap split (1-5)
   - Hitung rata-rata performance
   - Evaluasi generalization capability

3. **Gunakan file windowing_comparison.ipynb** untuk:
   - Testing windowing strategies
   - Model optimization
   - Performance comparison

**✅ Feature extraction completed! Ready untuk LSTM training dengan cross-validation.**